# Article 7 — Simulación WOM en Python

Abra este notebook en Google Colab. Cargue `Article_7_Python_v0.2.zip` en la siguiente celda. Primero ejecute las pruebas y el piloto; el experimento principal se activa explícitamente más adelante. Parámetros ilustrativos: estos resultados no son evidencia empírica.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, tempfile, subprocess, sys
uploaded = files.upload()
archive = next(name for name in uploaded if name == 'Article_7_Python_v0.2.zip')
work = Path(tempfile.mkdtemp(prefix='article7_'))
with zipfile.ZipFile(archive) as z:
    z.extractall(work)
project = work / 'Article_7_Python_v0.2'
print(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project/'requirements-tested.txt')], check=True)

## Verificación
Las 24 pruebas deben aprobar antes de ejecutar experimentos.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=project, check=True)

## Piloto — 70 simulaciones
Diez semillas emparejadas, siete políticas y 60 meses. Los directorios de salida deben estar vacíos. Para repetir, cambie el nombre del directorio.

In [ ]:
subprocess.run([sys.executable, '-m', 'article7.experiment', '--out', 'outputs/pilot', '--first-seed', '1001', '--replications', '10'], cwd=project, check=True)

## Experimento principal — 700 simulaciones
Active `RUN_MAIN` después de revisar el piloto. En la entrega original, ambas fases ya fueron ejecutadas. Aquí se reproducen con las mismas semillas.

In [ ]:
RUN_MAIN = False
if RUN_MAIN:
    subprocess.run([sys.executable, '-m', 'article7.experiment', '--out', 'outputs/main', '--first-seed', '2001', '--replications', '100'], cwd=project, check=True)
source = 'outputs/main' if RUN_MAIN else 'outputs/pilot'
subprocess.run([sys.executable, '-m', 'article7.analyze', '--source', source, '--out', 'outputs/analysis'], cwd=project, check=True)

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(project/source/'run_summary.csv').groupby('policy')[['new_customers','profit','adjusted_profit']].mean())
display(Image(filename=str(project/'outputs/analysis/baseline_policy_comparison.png')))

## Descargar el trabajo
Guarde el ZIP antes de cerrar Colab. Contiene datos originales, auditoría y análisis.

In [ ]:
import shutil
archive_path = shutil.make_archive(str(work/'Article_7_Resultados_reproducidos'), 'zip', project/'outputs')
files.download(archive_path)